In [1]:
from sklearn.neighbors import KNeighborsClassifier
import cv2
import pickle
import os
import csv
import time
import sqlite3
import numpy as np
import os
from datetime import datetime
from win32com.client import Dispatch

start_time = time.time()
date = datetime.now().strftime("%d-%m-%Y")
timestamp=datetime.now().strftime("%H:%M:%S")
times = datetime.now().strftime("%I:%M:%S %p")


def speak(str1):
    speak=Dispatch(("SAPI.SpVoice"))
    speak.Speak(str1)

DB_PATH = 'data/employees.db'

def get_connection():
    return sqlite3.connect(DB_PATH)


def load_faces_from_db():
    """
    Returns:
        faces         – np.ndarray of shape (N, 7500)
        names         – list of employee names (length N)
        employee_ids  – list of employee IDs (length N)
        emails        – list of emails (length N)
        organizations – list of organizations (length N)
    """
    conn   = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT name, employee_id, email, organization, face_vector FROM employees')
    rows   = cursor.fetchall()
    conn.close()

    names         = [r[0] for r in rows]
    employee_ids  = [r[1] for r in rows]
    emails        = [r[2] for r in rows]
    organizations = [r[3] for r in rows]
    faces         = np.array([
        np.frombuffer(r[4], dtype=np.uint8).astype(np.float64) / 255.0
        for r in rows
    ]) 

    return faces, names, employee_ids, emails, organizations

FACES, NAMES, EMPLOYEE_IDS, EMAILS, ORGANIZATIONS = load_faces_from_db()

print('Loaded faces shape:', FACES.shape)


knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(FACES, EMPLOYEE_IDS)


video=cv2.VideoCapture(0)
facedetect=cv2.CascadeClassifier('data/haarcascade_frontalface_default.xml')

imgBackground=cv2.imread("background.png")
while True:
    ret,frame=video.read()
    gray=cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces=facedetect.detectMultiScale(gray, 1.3 ,5)
    for (x,y,w,h) in faces:
        crop_img=frame[y:y+h, x:x+w, :]
        resized_img = cv2.resize(crop_img, (50, 50)).reshape(1, -1)/255.0
        
        distances, indices = knn.kneighbors(resized_img)
        threshold = 18                                                                      
        if distances[0][0] > threshold:
            # ── Unknown face ──────────────────────────────────────────────
            predicted_id    = None
            name            = 'Unknown'
            recipient_email = None
            organization    = None
        else:
            predicted_id = knn.predict(resized_img)[0] 

            if predicted_id in EMPLOYEE_IDS:
                index           = EMPLOYEE_IDS.index(predicted_id)
                name            = NAMES[index]
                recipient_email = EMAILS[index]
                organization    = ORGANIZATIONS[index]
            else:
                predicted_id    = None
                name            = 'Unknown'
                recipient_email = None
                organization    = None           
        cv2.rectangle(frame,(x,y),(x+w,y+h),(50,50,255),2)
        cv2.rectangle(frame,(x,y-35),(x+w,y),(0,128,255),-1)
        cv2.putText(frame, str(name), (x,y-15), cv2.FONT_HERSHEY_COMPLEX, 0.9, (255,255,255), 1)        
        imgBackground[162:162 + 480, 55:55 + 640] = frame
    cv2.imshow("Frame",imgBackground)
    k=cv2.waitKey(1)
    if k == ord('o') and predicted_id != None:
        speak("Attendance Taken..")
        time.sleep(1)
        exist=os.path.isfile("Attendance/Attendance_" + date + ".csv")
        COL_NAMES = ['EMPLOYEE_ID','NAME','TIME ', 'ORGANIZATION']
        attendance=[ str(predicted_id),str(name),str(timestamp), str(organization)]        
        if exist:
            with open("Attendance/Attendance_" + date + ".csv", "+a") as csvfile:
                writer=csv.writer(csvfile)
                writer.writerow(attendance)
            csvfile.close()
        else:
            with open("Attendance/Attendance_" + date + ".csv", "+a") as csvfile:
                writer=csv.writer(csvfile)
                writer.writerow(COL_NAMES)
                writer.writerow(attendance)
            csvfile.close()
    if k==ord('q') or (time.time() - start_time > 10):
        break
video.release()
cv2.destroyAllWindows()


Loaded faces shape: (50, 7500)
